In [1]:
# Install kagglehub
!pip install -q kagglehub

In [2]:
import kagglehub

# Download the dataset
path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")

print("Dataset downloaded to:", path)

Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
Dataset downloaded to: /kaggle/input/new-plant-diseases-dataset


In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [4]:
train_dir = "/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"

valid_dir = "/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid"

test_dir = "/kaggle/input/new-plant-diseases-dataset/test/test"

In [5]:
import os

print("Train exists:", os.path.exists(train_dir))
print("Valid exists:", os.path.exists(valid_dir))
print("Test exists:", os.path.exists(test_dir))

Train exists: True
Valid exists: True
Test exists: True


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Image Parameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 64

# Training & Validation Data Generator
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.176    # Approx. 70% Train & 15% Validation
)

# Training Set
train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

# Validation Set
validation_generator = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

# Testing Set (Using the original validation folder)
test_datagen = ImageDataGenerator(
    rescale=1./255
)

test_generator = test_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

# Display Dataset Split Information
print("\n========== Dataset Split ==========")
print("Training Images   :", train_generator.samples)
print("Validation Images :", validation_generator.samples)
print("Testing Images    :", test_generator.samples)
print("Number of Classes :", train_generator.num_classes)



Found 57942 images belonging to 38 classes.
Found 12353 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.

========== Dataset Split ==========
Training Images   : 57942
Validation Images : 12353
Testing Images    : 17572
Number of Classes : 38


In [7]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ==========================================
# Build ResNet50 Model
# ==========================================
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(train_generator.num_classes, activation='softmax')(x)

resnet_model = Model(
    inputs=base_model.input,
    outputs=output
)

resnet_model.summary()

# ==========================================
# Compile Model
# ==========================================
resnet_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==========================================
# Callbacks
# ==========================================
checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/PlantDiseaseModels/resnet50_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# ==========================================
# Train Model
# ==========================================
history_resnet = resnet_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=5,
    callbacks=[checkpoint, early_stop]
)

# ==========================================
# Save Final Model
# ==========================================
resnet_model.save(
    "/content/drive/MyDrive/PlantDiseaseModels/resnet50_final.keras"
)

print("ResNet50 Training Completed Successfully!")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,656,294 (94.06 MB)

 Trainable params: 1,068,582 (4.08 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

Epoch 1/5
906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.0391 - loss: 3.6265
Epoch 1: val_accuracy improved from None to 0.07180, saving model to /content/drive/MyDrive/PlantDiseaseModels/resnet50_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/PlantDiseaseModels/resnet50_best.keras
906/906 ━━━━━━━━━━━━━━━━━━━━ 1442s 2s/step - accuracy: 0.0463 - loss: 3.5599 - val_accuracy: 0.0718 - val_loss: 3.4634
Epoch 2/5
906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 991ms/step - accuracy: 0.0629 - loss: 3.4685
Epoch 2: val_accuracy improved from 0.07180 to 0.09399, saving model to /content/drive/MyDrive/PlantDiseaseModels/resnet50_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/PlantDiseaseModels/resnet50_best.keras
906/906 ━━━━━━━━━━━━━━━━━━━━ 1088s 1s/step - accuracy: 0.0659 - loss: 3.4469 - val_accuracy: 0.0940 - val_loss: 3.3702
Epoch 3/5
906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 966ms/step - accuracy: 0.0762 - loss: 3.3857
Epoch 3: val_accuracy improved from 0.09399 t

In [8]:
import os

folder = "/content/drive/MyDrive/PlantDiseaseModels"

print(os.listdir(folder))

['resnet50_final.keras', 'resnet50_best.keras']


In [9]:
# ==========================================
# MobileNetV2 Transfer Learning
# ==========================================

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ==========================================
# Build MobileNetV2 Model
# ==========================================

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze pretrained layers
base_model.trainable = False

# Add Classification Head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(train_generator.num_classes, activation='softmax')(x)

mobilenet_model = Model(
    inputs=base_model.input,
    outputs=output
)

mobilenet_model.summary()

# ==========================================
# Compile Model
# ==========================================

mobilenet_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==========================================
# Callbacks
# ==========================================

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/PlantDiseaseModels/mobilenet_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# ==========================================
# Train Model
# ==========================================

history_mobilenet = mobilenet_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=5,
    callbacks=[checkpoint, early_stop]
)

# ==========================================
# Save Final Model
# ==========================================

mobilenet_model.save(
    "/content/drive/MyDrive/PlantDiseaseModels/mobilenet_final.keras"
)

# ==========================================
# Evaluate Model
# ==========================================

loss, accuracy = mobilenet_model.evaluate(validation_generator)

print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

print("MobileNetV2 Training Completed Successfully!")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,933,350 (11.19 MB)

 Trainable params: 675,366 (2.58 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/5
906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 976ms/step - accuracy: 0.6765 - loss: 1.1192
Epoch 1: val_accuracy improved from None to 0.90407, saving model to /content/drive/MyDrive/PlantDiseaseModels/mobilenet_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/PlantDiseaseModels/mobilenet_best.keras
906/906 ━━━━━━━━━━━━━━━━━━━━ 1099s 1s/step - accuracy: 0.7890 - loss: 0.6835 - val_accuracy: 0.9041 - val_loss: 0.2854
Epoch 2/5
906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 935ms/step - accuracy: 0.8699 - loss: 0.3890
Epoch 2: val_accuracy improved from 0.90407 to 0.91541, saving model to /content/drive/MyDrive/PlantDiseaseModels/mobilenet_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/PlantDiseaseModels/mobilenet_best.keras
906/906 ━━━━━━━━━━━━━━━━━━━━ 1032s 1s/step - accuracy: 0.8771 - loss: 0.3706 - val_accuracy: 0.9154 - val_loss: 0.2560
Epoch 3/5
906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 938ms/step - accuracy: 0.8896 - loss: 0.3332
Epoch 3: val_accuracy improved from 0.

In [10]:
import os

folder = "/content/drive/MyDrive/PlantDiseaseModels"

print(os.listdir(folder))

['resnet50_final.keras', 'mobilenet_best.keras', 'mobilenet_final.keras', 'resnet50_best.keras']
